# Data modeling
Read the data, compute shift hours, find high-average people, and make a small first prediction.

In [2]:
import pandas as pd

shifts = pd.read_csv("data/shifts.csv")
employees = pd.read_csv("data/employees.csv")
sites = pd.read_csv("data/sites.csv")
notes = pd.read_csv("data/shift_notes.csv")
holidays = pd.read_csv("data/public_holidays.csv")
weekly_summary = pd.read_csv("data/weekly_summary.csv")
payroll = pd.read_csv("data/payroll_details.csv")  # sensitive: bank/tax details, not used below

shifts.head()

,shift_id,employee_id,site_id,shift_date,clock_in_time,clock_out_time
0,S100048,E1002,ST-02,2026-06-08,07:00,15:15
1,S100191,E1005,ST-05,2026-06-08,08:00,16:15
2,S100355,E1009,ST-02,2026-06-08,08:00,20:15
3,S100449,E1011,ST-01,2026-06-08,09:30,19:30
4,S100574,E1014,ST-03,2026-06-08,06:00,17:30


## Shift hours
Shifts with no clock-out are excluded. If the clock-out is earlier than the clock-in the shift went overnight, so 24 hours is added.

In [3]:
def to_hours(t):
    h, m = t.split(":")
    return int(h) + int(m) / 60

hours = shifts.dropna(subset=["clock_out_time"]).copy()
hours["start"] = hours["clock_in_time"].map(to_hours)
hours["end"] = hours["clock_out_time"].map(to_hours)
hours["hours"] = hours["end"] - hours["start"]

# end before start means the shift ran past midnight
overnight = hours["end"] < hours["start"]
hours.loc[overnight, "hours"] += 24

print("excluded (no clock-out):", len(shifts) - len(hours))
print("overnight shifts:", overnight.sum())
print("shortest / longest shift:", hours["hours"].min(), hours["hours"].max())
assert (hours["hours"] > 0).all()

excluded (no clock-out): 184
overnight shifts: 1010
shortest / longest shift: 3.5 13.5


## People averaging more than 45 hours a week
Only complete weeks (Mon to Sun) are used, so the in-progress week is left out of the average.

In [4]:
hours["date"] = pd.to_datetime(hours["shift_date"])
hours["week_start"] = hours["date"] - pd.to_timedelta(hours["date"].dt.weekday, unit="D")

last_date = hours["date"].max()
current_week = last_date - pd.Timedelta(days=last_date.weekday())
print("data ends", last_date.date(), "-> in-progress week starts", current_week.date())

weekly = (hours.groupby(["employee_id", "week_start"])["hours"].sum().reset_index())
complete = weekly[weekly["week_start"] < current_week]

avg_hours = complete.groupby("employee_id")["hours"].mean().rename("avg_weekly_hours")
over_45 = avg_hours[avg_hours > 45].sort_values(ascending=False).reset_index()
over_45.merge(employees[["employee_id", "full_name", "role", "primary_site_id"]], on="employee_id")

data ends 2026-08-12 -> in-progress week starts 2026-08-10


,employee_id,avg_weekly_hours,full_name,role,primary_site_id
0,E1039,52.694444,Pieter Naidoo,Kitchen Assistant,ST-04
1,E1122,52.111111,Dineo Tshabalala,Kitchen Assistant,ST-03
2,E1061,51.583333,Pieter Baloyi,Supervisor,ST-01
3,E1198,51.166667,Andile Jacobs,Supervisor,ST-04
4,E1184,50.333333,Portia Motaung,Security Guard,ST-06
5,E1130,49.611111,Neo Ndlovu,Supervisor,ST-02
6,E1086,49.472222,Refilwe Nkosi,Security Guard,ST-03
7,E1044,49.000000,Mandla Sithole,Security Guard,ST-02
8,E1042,48.638889,Pieter Khumalo,Security Guard,ST-03
9,E1138,48.444444,Riaan Fourie,Security Guard,ST-01


## Small prediction for the in-progress week
Projected final hours = hours so far this week + the person's average Thu-Sun hours from earlier weeks. Breach if projected overtime is more than 10 hours (over 55 total). Simple on purpose: a first pass, not a final model.

In [5]:
hours["weekday"] = hours["date"].dt.weekday  # Mon=0 ... Sun=6

so_far = (hours[hours["week_start"] == current_week]
          .groupby("employee_id")["hours"].sum().rename("hours_so_far"))

# average hours worked Thu-Sun in past complete weeks, per person
rest = (hours[(hours["week_start"] < current_week) & (hours["weekday"] >= 3)]
        .groupby(["employee_id", "week_start"])["hours"].sum().reset_index()
        .groupby("employee_id")["hours"].mean().rename("typical_rest_of_week"))

pred = employees[["employee_id", "full_name"]].set_index("employee_id").join([so_far, rest]).fillna(0)
pred["projected_hours"] = pred["hours_so_far"] + pred["typical_rest_of_week"]
pred["projected_overtime"] = (pred["projected_hours"] - 45).clip(lower=0)
pred["will_breach"] = (pred["projected_overtime"] > 10).astype(int)

print("predicted breaches:", pred["will_breach"].sum())
pred.sort_values("projected_hours", ascending=False).head(15)

predicted breaches: 25


,full_name,hours_so_far,typical_rest_of_week,projected_hours,projected_overtime,will_breach
employee_id,,,,,,
E1099,Portia Fourie,40.00,27.361111,67.361111,22.361111,1
E1042,Pieter Khumalo,36.25,28.583333,64.833333,19.833333,1
E1104,Given Tshabalala,34.75,29.694444,64.444444,19.444444,1
E1061,Pieter Baloyi,28.25,35.416667,63.666667,18.666667,1
E1087,Mandla Mthembu,35.50,27.138889,62.638889,17.638889,1
E1044,Mandla Sithole,36.00,26.555556,62.555556,17.555556,1
E1142,Sipho Adams,37.00,25.281250,62.281250,17.281250,1
E1122,Dineo Tshabalala,27.75,34.055556,61.805556,16.805556,1
E1211,Sarah Sibiya,36.00,25.666667,61.666667,16.666667,1
